In [1]:
# Colab 셀 1: 필수 라이브러리 설치
!pip install datasets openai tqdm -q
print("✅ 라이브러리 설치 완료!")

✅ 라이브러리 설치 완료!


In [5]:
# Colab 셀 2: 데이터셋 번역 스크립트
import os
import json
from google.colab import userdata
from openai import OpenAI
from datasets import load_dataset
from tqdm.auto import tqdm

# --- 1. 설정 ---
# Colab 비밀 관리자에서 API 키 가져오기
try:
    api_key = userdata.get('OPENAI_API_KEY')
    os.environ['OPENAI_API_KEY'] = api_key
    client = OpenAI()
    print("API 키가 성공적으로 로드되었습니다.")
except Exception as e:
    print(f"API 키 로드 실패: {e}\nColab의 '비밀'에 OPENAI_API_KEY를 설정했는지 확인하세요.")
    client = None

# 번역할 데이터셋 정보
DATASET_NAME = "javirandor/hh-rlhf-safety-v3-dpo"
DATASET_SPLIT = "train"

# 결과 저장 파일
OUTPUT_FILE = "translated_safety_dpo_ko.jsonl"
ERROR_LOG_FILE = "translation_errors.log"

# 중간 저장 간격 (몇 개 레코드마다 저장할지)
SAVE_INTERVAL = 50

# --- 2. 번역 함수 정의 ---
def translate_record(record):
    """하나의 레코드(prompt, chosen, rejected)를 받아 번역된 결과를 반환합니다."""
    # GPT에 전달할 JSON 형식의 내용 구성
    content_to_translate = {
        "prompt": record["prompt"],
        "chosen": record["chosen_response"],
        "rejected": record["rejected_response"]
    }

    system_prompt = "You are an expert translator specializing in AI safety. Translate the string values in the following JSON object from English to Korean. Maintain the original JSON structure and keys."

    response = client.chat.completions.create(
        model="gpt-4.1-mini",  # 또는 "gpt-4-turbo"
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(content_to_translate, ensure_ascii=False)}
        ],
        response_format={"type": "json_object"},
        temperature=0.3
    )

    translated_content = json.loads(response.choices[0].message.content)
    return translated_content

# --- 3. 메인 실행 로직 ---
if client:
    # 1. 원본 데이터셋 로드
    print(f"원본 데이터셋 로드 중: {DATASET_NAME}")
    dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)

    # 테스트를 위해 일부 데이터만 사용하려면 아래 주석을 해제하세요.
    dataset = dataset.select(range(10))
    print("⚠️ 테스트 모드: 10개의 데이터만 처리합니다.")

    # 2. 이어하기를 위한 기존 데이터 로드
    translated_records = []
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                translated_records.append(json.loads(line))
        print(f"기존에 번역된 {len(translated_records)}개의 데이터를 로드했습니다. 이어서 작업을 시작합니다.")

    start_index = len(translated_records)

    # 3. 번역 루프 실행
    with open(OUTPUT_FILE, 'a', encoding='utf-8') as f_out:
        for i in tqdm(range(start_index, len(dataset)), desc="데이터셋 번역 중"):
            try:
                original_record = dataset[i]
                translated_record = translate_record(original_record)

                # 파일에 바로 저장 (JSON Lines 형식)
                f_out.write(json.dumps(translated_record, ensure_ascii=False) + '\n')

                # 중간 저장 시점을 알리기 위한 print (선택사항)
                if (i + 1) % SAVE_INTERVAL == 0:
                    print(f"\n진행 상황: {i + 1}/{len(dataset)}개 번역 완료 및 저장됨.")

            except Exception as e:
                error_message = f"Error processing record {i}: {e}"
                print(f"\n{error_message}")
                with open(ERROR_LOG_FILE, 'a', encoding='utf-8') as f_err:
                    f_err.write(error_message + '\n')

    print(f"🎉 번역 작업 완료! 결과는 '{OUTPUT_FILE}' 파일에 저장되었습니다.")

API 키가 성공적으로 로드되었습니다.
원본 데이터셋 로드 중: javirandor/hh-rlhf-safety-v3-dpo
⚠️ 테스트 모드: 10개의 데이터만 처리합니다.


데이터셋 번역 중:   0%|          | 0/10 [00:00<?, ?it/s]

🎉 번역 작업 완료! 결과는 'translated_safety_dpo_ko.jsonl' 파일에 저장되었습니다.


In [ ]:
from datasets import load_dataset, concatenate_datasets

# 번역된 safety 데이터 로드
translated_safety_ds = load_dataset('json', data_files='translated_safety_dpo_ko.jsonl', split='train')

# 금융 데이터 로드
finance_ds = load_dataset("aiqwe/FinShibainu", split="qa")

# (필요시) 금융 데이터 형식 변환 (prompt, chosen, rejected)
# finance_ds = finance_ds.map(...)

# 두 데이터셋 병합
final_dpo_dataset = concatenate_datasets([translated_safety_ds, finance_ds])

print(final_dpo_dataset)